### Multisession registration with CaImAn

This notebook is modified from the demo notebook: https://github.com/flatironinstitute/CaImAn/blob/main/demos/notebooks/demo_multisession_registration.ipynb

This notebook is used to register recording over multiple sessions. CaImAn has in-built functions that align movies from two or more sessions and try to recognize components that are imaged in some or all of these recordings.

The basic function for this is `caiman.base.rois.register_ROIs()`. It takes two sets of spatial components and finds components present in both using an intersection over union metric and the Hungarian algorithm for optimal matching.
`caiman.base.rois.register_multisession()` takes a list of spatial components, aligns sessions 1 and 2, keeps the union of the matched and unmatched components to register it with session 3 and so on.

#### Import necessary libraries

In [ ]:
from IPython import get_ipython
import numpy as np
import json
import scipy.io as sio
import os
from caiman.base.rois import register_multisession
from caiman.utils import visualization

try:
    if __IPYTHON__:
        get_ipython().run_line_magic('load_ext', 'autoreload')
        get_ipython().run_line_magic('autoreload', '2')
except NameError:
    pass

#### Define a function to load and export paths

In [ ]:
def load_json_and_extract_export_paths(json_file_path):
    try:
        # Load JSON file
        with open(json_file_path, 'r') as file:
            data = json.load(file)
        
        # Check if 'export_paths' field exists in JSON file
        if 'export_paths' in data:
            export_paths = data['export_paths']
            return export_paths
        else:
            print(f"JSON file does not contain 'export_paths' field.")
            return None
    
    except FileNotFoundError:
        print(f"JSON file not found at path: {json_file_path}")
        return None
    
    except json.JSONDecodeError:
        print(f"Invalid JSON format in file: {json_file_path}")
        return None

#### Define paths

In [ ]:
#  Load export from json file
# json_file_path = '/home/wanglab/code/imaging_analysis/Analysis_2P/test/paths_C57_O1M2_10022023_run5run6run7run8run9run10_linux.json'
# export_paths = load_json_and_extract_export_paths(json_file_path)
# Or manually set export_paths
export_paths = ['H:/Analysis/2P/C57_O1M2/10022023/run1/mesmerize',
                'H:/Analysis/2P/C57_O1M2/10022023/run2/mesmerize',
                'H:/Analysis/2P/C57_O1M2/10022023/run3/mesmerize',
                'H:/Analysis/2P/C57_O1M2/10022023/run4/mesmerize']

# if export_paths is not None:
#     print(export_paths)

mat_file_paths = [path + '/results_caiman.mat' for path in export_paths]
cn_paths=[path + '/mean_intensity_template.npy' for path in export_paths]
print(mat_file_paths)
print(cn_paths)

In [ ]:
# Create lists to store spatial components and templates
spatial=[]
templates=[]
for path in mat_file_paths:
    spatial.append(sio.loadmat(path)['spatial_components'])

for path in cn_paths:
    # Check if file exists
    if not os.path.exists(path):
        #  skip if file does not exist
        break
    templates.append(np.load(path, allow_pickle=True)) 

# If templates is empty, load from mat file
if not templates:
    for path in mat_file_paths:
        templates.append(sio.loadmat(path)['mean_map_motion_corrected'])

dims = templates[0].shape

#### Use `register_multisession()`

The function `register_multisession()` requires 3 arguments:
- `A`: A list of ndarrays or scipy.sparse.csc matrices with (# pixels X # component ROIs) for each session
- `dims`: Dimensions of the FOV, needed to restore spatial components to a 2D image
- `templates`: List of ndarray matrices of size `dims`, template image of each session

In [ ]:
spatial_union, assignments, matchings = register_multisession(A=spatial, dims=dims, templates=templates)

The function returns 3 variables for further analysis:
- `spatial_union`: csc_matrix (# pixels X # total distinct components), the union of all ROIs across all sessions aligned to the FOV of the last session.
- `assignments`: ndarray (# total distinct components X # sessions). `assignments[i,j]=k` means that component `k` from session `j` has been identified as component `i` from the union of all components, otherwise it takes a `NaN` value. Note that for each `i` there is at least one session index `j` where `assignments[i,j]!=NaN`.
- `matchings`: list of (# sessions) lists. Saves `spatial_union` indices of individual components in each session. `matchings[j][k] = i` means that component `k` from session `j` is represented by component `i` in the union of all components `spatial_union`. In other words `assignments[matchings[j][k], j] = j`.

#### Post-alignment screening

The three outputs can be used to filter components in various ways. For example we can find the components that were active in at least a given a number of sessions. For more examples, check [this script](https://github.com/flatironinstitute/CaImAn/blob/master/use_cases/eLife_scripts/figure_9/Figure_9_alignment.py) that reproduces the results of [Figure 9, as presented in our eLife paper](https://elifesciences.org/articles/38173#fig9).

In [ ]:
# Filter components by number of sessions the component could be found

# n_reg = 3 # minimal number of sessions that each component has to be registered in
n_reg=len(export_paths)
# Use number of non-NaNs in each row to filter out components that were not registered in enough sessions
assignments_filtered = np.array(np.nan_to_num(assignments[np.sum(~np.isnan(assignments), axis=1) >= n_reg]), dtype=int);

# Use filtered indices to select the corresponding spatial components
spatial_filtered = spatial[0][:, assignments_filtered[:, 0]]

# Plot spatial components of the selected components on the template of the last session
visualization.plot_contours(spatial_filtered, templates[-1])

#### Save assignments and matchings to the common directory

In [ ]:
from scipy.sparse import csc_matrix
root_dir = os.path.commonpath(export_paths)

# Convert the dense matrix to a sparse matrix
spatial_union_sparse = csc_matrix(spatial_union)

# Save to mat file
sio.savemat(root_dir + '/register_multisession.mat', mdict={'assignments': assignments, 'matchings': matchings, 'spatial_union': spatial_union_sparse, 'assignments_filtered': assignments_filtered})


#### Combining data of components over multiple sessions (optional)

Now that all sessions are aligned and we have a list of re-registered neurons, we can use `assignments` and `matchings` to collect traces from neurons over different sessions.

As an exercise, we can collect the traces of all neurons that were registered in all sessions. We already gathered the indices of these neurons in the previous cell in `assignments_filtered`. Assuming that traces of each session are saved in their own `CNMF` object collected in a list, we can iterate through `assignments_filtered` and use these indices to find the re-registered neurons in every session.

Note: This notebook does not include the traces of the extracted neurons, only their spatial components. As such the loop below will produce an error if you uncomment it. However, it demonstrates how to use the results of the registration to in your own analysis to extract the traces of the same neurons across different sessions.

In [ ]:
# traces = np.zeros(assignments_filtered.shape, dtype=np.ndarray)
# for i in range(traces.shape[0]):
#     for j in range(traces.shape[1]):
#         traces[i,j] = cnm_list[j].estimates.C[int(assignments_filtered[i,j])]

Now we have the array `traces`, where element `traces[i,j] = k` is the temporal component of neuron `i` at session `j`. This can be performed with `F_dff` data or `S` spikes as well.